<a href="https://colab.research.google.com/github/danasapir/BME3053C-Spring-2025/blob/main/HWs/Homework_07.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Homework 07

import numpy as np
import matplotlib.pyplot as plt
from scipy.fftpack import fft, ifft
from scipy.signal import find_peaks

# Part 1: Generate and Visualize ECG Signal
def generate_ecg_data(duration, sampling_rate=250):
    def generate_ecg_pulse(duration, sampling_rate):
        t = np.linspace(0, duration, int(duration * sampling_rate), endpoint=False)
        p_wave = 0.15 * np.sin(2 * np.pi * 1.5 * t) * np.exp(-((t - 0.1) ** 2) / 0.005)
        qrs_complex = np.zeros_like(t)
        qrs_complex[(t > 0.2) & (t < 0.25)] = -0.3
        qrs_complex[(t > 0.25) & (t < 0.3)] = 1.0
        qrs_complex[(t > 0.3) & (t < 0.35)] = -0.2
        t_wave = 0.3 * np.sin(2 * np.pi * 0.75 * (t - 0.45)) * np.exp(-((t - 0.45) ** 2) / 0.015)
        pulse = p_wave + qrs_complex + t_wave
        high_freq_noise = 0.05 * np.sin(2 * np.pi * 50 * t)
        pulse += high_freq_noise
        return pulse

    single_pulse = generate_ecg_pulse(0.8, sampling_rate)
    num_pulses = int(duration / 0.8)
    ecg_data = np.tile(single_pulse, num_pulses)
    noise = np.random.normal(0, 0.05, len(ecg_data))
    ecg_data += noise
    t = np.linspace(0, duration, len(ecg_data), endpoint=False)
    baseline_wander = 0.1 * np.sin(2 * np.pi * 0.1 * t)
    ecg_data += baseline_wander
    power_line_interference = 0.05 * np.sin(2 * np.pi * 60 * t)
    ecg_data += power_line_interference
    ecg_data = ecg_data[:int(duration * sampling_rate)]
    return ecg_data

ecg_data = generate_ecg_data(10, 250)
sampling_rate = 250
t = np.arange(len(ecg_data)) / sampling_rate

plt.figure(figsize=(10, 4))
plt.plot(t, ecg_data)
plt.title('Raw ECG Signal')
plt.xlabel('Time (s)')
plt.ylabel('ECG Amplitude')
plt.grid(True)
plt.savefig('raw_ecg.png')
plt.close()

# Part 2: Fourier Transform
N = len(ecg_data)
fft_data = fft(ecg_data)
freq = np.fft.fftfreq(N, 1/sampling_rate)
positive_freq_mask = freq >= 0
freq = freq[positive_freq_mask]
fft_magnitude = np.abs(fft_data)[:N//2] / N

plt.figure(figsize=(10, 4))
plt.plot(freq, fft_magnitude)
plt.title('Fourier Transform of ECG Signal')
plt.xlabel('Frequency (Hz)')
plt.ylabel('Magnitude')
plt.grid(True)
plt.savefig('fft_ecg.png')
plt.close()

# Part 3: Bandpass Filter
freq_mask = (np.abs(freq) >= 0.5) & (np.abs(freq) <= 40)
filtered_fft = fft_data.copy()
filtered_fft[int(N/2):] = filtered_fft[:N//2][::-1]  # Ensure symmetry
filtered_fft[~freq_mask] = 0  # Apply mask to positive and negative frequencies
filtered_ecg = ifft(filtered_fft).real

plt.figure(figsize=(10, 4))
plt.plot(t, filtered_ecg)
plt.title('Filtered ECG Signal')
plt.xlabel('Time (s)')
plt.ylabel('ECG Amplitude')
plt.grid(True)
plt.savefig('filtered_ecg.png')
plt.close()

# Part 4: Heart Rate Calculation
peaks, _ = find_peaks(filtered_ecg, distance=sampling_rate*0.5, prominence=0.5)
peak_times = t[peaks]
intervals = np.diff(peak_times)
avg_interval = np.mean(intervals)
heart_rate = 60 / avg_interval

plt.figure(figsize=(10, 4))
plt.plot(t, filtered_ecg)
plt.plot(t[peaks], filtered_ecg[peaks], 'ro')
plt.title(f'Filtered ECG with R-Peaks (Heart Rate: {heart_rate:.1f} BPM)')
plt.xlabel('Time (s)')
plt.ylabel('ECG Amplitude')
plt.grid(True)
plt.savefig('r_peaks.png')
plt.close()

# Part 5: Summary (to be displayed in notebook, not plotted)
summary = f"""
Analysis of the ECG signal revealed key insights through signal processing techniques.
The raw ECG signal exhibited noise, including baseline wander and power line interference, which was evident in the time-domain plot.
The Fourier transform highlighted frequency components, with significant magnitudes around 0.5–40 Hz, corresponding to physiological ECG features.
Applying a bandpass filter (0.5–40 Hz) effectively removed low-frequency baseline wander and high-frequency noise, resulting in a cleaner signal.
R-peak detection on the filtered signal yielded an average heart rate of approximately {heart_rate:.1f} BPM, consistent with the generated data's pulse repetition rate.
These steps demonstrate the power of Fourier analysis and filtering in extracting meaningful physiological information from noisy biomedical signals.
"""

with open('summary.txt', 'w') as f:
    f.write(summary)